# Figure 11_r — Maize Heat-Wave RGB Maps (Mean Only)
### DSSAT vs WOFOST  |  1-D Bivariate Colour Mapping

| Colour | Meaning |
|--------|---------|
| **Magenta** | DSSAT shows stronger yield change |
| **Green** | WOFOST shows stronger yield change |
| **White** | Models agree (near-zero relative difference) |

Single-panel figure: **Mean percent-difference map** only.

## 1. Configuration & Imports

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

mpl.rcParams['font.family'] = 'Arial'

# ── Data paths ──────────────────────────────────────────────
_nb_dir  = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.normpath(os.path.join(_nb_dir, '..', 'data'))

PATHS = {
    'city_mask':     os.path.join(DATA_DIR, '4k_grid_city_only.geojson'),
    'temp_shp':      os.path.join(DATA_DIR, 'COMBINED_DIST.geojson'),
    'us_states':     os.path.join(DATA_DIR, 'us_states_for_map.geojson'),
    'counties_usa':  os.path.join(DATA_DIR, 'counties_usa_for_map.geojson'),
    'ag_60':         os.path.join(DATA_DIR, 'ag_dis_n.geojson'),
    'ag_90':         os.path.join(DATA_DIR, 'ag_dis_s.geojson'),
    'combined_dist': os.path.join(DATA_DIR, 'COMBINED_DIST_with_county .geojson'),
    'grid':          os.path.join(DATA_DIR, '4km_2025_grid_pol.geojson'),
}

DSSAT_DIR  = os.path.join(DATA_DIR, 'DSSAT_Maize')
WOFOST_DIR = os.path.join(DATA_DIR, 'WOFOST_Maize')

OUTPUT_DIR = os.path.join(DATA_DIR, 'output_maps_maize_fig_11')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Plot styling ───────────────────────────────────────────
STYLE = {
    'markersize':       3,
    'marker_lw':        1.5,
    'lw_states':        2.5,
    'lw_counties':      0.5,
    'lw_boundary':      2.0,
    'color_borders':    'black',
    'color_counties':   'gray',
    'facecolor_bg':     'lightgray',
    'city_face':        '#222222',
    'city_edge':        'black',
    'zoom_main':        0.20,
    'zoom_inset':       10,
    'fig_size':         (10, 8),
    'dpi':              300,
}

print(f'DATA_DIR  = {DATA_DIR}')
print(f'OUTPUT    = {OUTPUT_DIR}')
print('Configuration OK')

DATA_DIR  = c:\BioCro_DSSAT_WOFOST_Egorov_etal_paper-main\data
OUTPUT    = c:\BioCro_DSSAT_WOFOST_Egorov_etal_paper-main\data\output_maps_maize_fig11
Configuration OK


## 2. Helper Functions

In [9]:
# ── Bivariate colour endpoints ─────────────────────────────
C_MAGENTA = np.array([255,   0, 255]) / 255.0   # DSSAT dominant
C_GREEN   = np.array([0,   255,   0]) / 255.0   # WOFOST dominant


def dy_to_rgb(dy, sigma=0.05):
    """1-D bivariate Gaussian blend:  Magenta (-1) -> White (0) -> Green (+1)."""
    wy   = np.clip((dy + 1) / 2, 0, 1)[..., None]
    base = (1 - wy) * C_MAGENTA + wy * C_GREEN
    w0   = np.exp(-(dy ** 2) / sigma ** 2)[..., None]
    return (1 - w0) * base + w0 * 1.0


def rgb_to_hex_safe(rgb_arr, nan_colour='#cccccc'):
    """Convert (N, 3) float RGB array to flat array of hex strings."""
    flat = np.reshape(rgb_arr, (-1, 3))
    out  = []
    for triplet in flat:
        if np.isnan(triplet).any():
            out.append(nan_colour)
        else:
            r, g, b = np.clip(triplet * 255, 0, 255).astype(int)
            out.append(f'#{r:02x}{g:02x}{b:02x}')
    return np.array(out)


def set_zoom(ax, gdf, padding):
    """Set axis limits from GeoDataFrame bounds + fractional padding."""
    xmin, ymin, xmax, ymax = gdf.total_bounds
    dx, dy = (xmax - xmin) * padding, (ymax - ymin) * padding
    ax.set_xlim(xmin - dx, xmax + dx)
    ax.set_ylim(ymin - dy, ymax + dy)


def add_inset(parent_ax, district_gdf, states_gdf, ref_gdf, zoom_pad):
    """Add a small locator inset map to the upper-right of *parent_ax*."""
    axins = inset_axes(parent_ax, width=1.5, height=1.5, loc='upper right',
                       borderpad=0, bbox_to_anchor=(1.00, 1.033),
                       bbox_transform=parent_ax.transAxes)
    district_gdf.to_crs(epsg=4326).plot(ax=axins, color='black',
                                         edgecolor='black', alpha=0.5)
    xmin, ymin, xmax, ymax = ref_gdf.total_bounds
    xp = (xmax - xmin) * zoom_pad
    yp = (ymax - ymin) * zoom_pad
    axins.set_xlim(xmin - xp, xmax + xp)
    axins.set_ylim(ymin - yp, ymax + yp)
    states_gdf.to_crs(epsg=4326).plot(ax=axins, color='lightgray',
                                       edgecolor='black', alpha=1.0)
    ref_gdf.plot(ax=axins, color='lightgray', edgecolor='red', alpha=1.0)
    axins.set_xticks([]); axins.set_yticks([]); axins.set_title('')
    return axins


def compute_diffs(df_a, df_b):
    """Year-by-year diff & percent-diff between two GeoDataFrames.
    Returns a GeoDataFrame with mean_difference, mean_percent_difference,
    mean_difference_STD columns."""
    year_cols_a = {c[-4:]: c for c in df_a.columns if c[-4:].isdigit()}
    year_cols_b = {c[-4:]: c for c in df_b.columns if c[-4:].isdigit()}
    common = sorted(set(year_cols_a) & set(year_cols_b))

    result = pd.DataFrame({'real_id': df_a['real_id']})
    diff_cols, pct_cols = [], []

    for yr in common:
        ca, cb = year_cols_a[yr], year_cols_b[yr]
        diff = df_a[ca] - df_b[cb]
        pct  = ((diff / df_b[cb]) * 100).replace([np.inf, -np.inf], pd.NA)
        result[f'diff_{yr}']    = diff
        result[f'percent_{yr}'] = pct
        diff_cols.append(f'diff_{yr}')
        pct_cols.append(f'percent_{yr}')

    result['mean_difference']         = result[diff_cols].mean(axis=1)
    result['mean_percent_difference'] = result[pct_cols].mean(axis=1)
    result['mean_difference_STD']     = result[pct_cols].std(axis=1)

    result['Col_A_pattern'] = year_cols_a[common[0]] if common else None
    result['Col_B_pattern'] = year_cols_b[common[0]] if common else None

    result = result.merge(df_a[['real_id', 'geometry']], on='real_id')
    return gpd.GeoDataFrame(result, geometry='geometry', crs=df_a.crs)


def process_yield_data(input_directory):
    """Read per-year CSVs (skip 2000), merge on real_id, return summary CSV path."""
    dir_name    = os.path.basename(os.path.normpath(input_directory))
    output_file = os.path.join(input_directory,
                               f'{dir_name}_grain_yield_mean_with_years.csv')
    if os.path.exists(output_file):
        os.remove(output_file)

    csvs = [f for f in glob.glob(os.path.join(input_directory, '*.csv'))
            if '2000' not in os.path.basename(f)]
    if not csvs:
        print(f'  [!] No CSVs in {input_directory}')
        return None

    frames = []
    for fp in csvs:
        try:
            df = pd.read_csv(fp)
            if 'real_id' not in df.columns and 'id' in df.columns:
                df.rename(columns={'id': 'real_id'}, inplace=True)
            if 'real_id' not in df.columns:
                continue
            ycol = 'Grain' if 'Grain' in df.columns else (
                   'yield' if 'yield' in df.columns else None)
            if ycol is None:
                continue
            suffix = os.path.splitext(os.path.basename(fp))[0].replace('.', '_')
            df = df[['real_id', ycol]].rename(columns={ycol: f'{ycol}_{suffix}'})
            frames.append(df)
        except Exception as e:
            print(f'  [!] Error reading {fp}: {e}')

    if not frames:
        return None

    merged = frames[0]
    for f in frames[1:]:
        merged = pd.merge(merged, f, on='real_id', how='outer')

    ycols = [c for c in merged.columns if c.startswith('Grain_') or c.startswith('yield_')]
    merged['Yield_mean'] = merged[ycols].mean(axis=1)
    merged['Yield_std']  = merged[ycols].std(axis=1)
    merged.to_csv(output_file, index=False)
    return output_file


print('Helper functions loaded')

Helper functions loaded


## 3. Load Spatial Layers

In [10]:
city_mask    = gpd.read_file(PATHS['city_mask']).set_crs(epsg=4326)
temp_shp     = gpd.read_file(PATHS['temp_shp']).to_crs(epsg=4326)
us_states    = gpd.read_file(PATHS['us_states']).to_crs(epsg=4326)
counties_usa = gpd.read_file(PATHS['counties_usa']).to_crs(epsg=4326)
district_60  = gpd.read_file(PATHS['ag_60']).set_crs(epsg=4326).to_crs(epsg=4326)
district_90  = gpd.read_file(PATHS['ag_90']).set_crs(epsg=4326).to_crs(epsg=4326)
ref_districts = gpd.read_file(PATHS['combined_dist']).set_crs(epsg=4326).to_crs(epsg=4326)

grid_file = gpd.read_file(PATHS['grid']).set_crs(epsg=4326).to_crs(epsg=4326)
grid_file['Id'] = grid_file['ORIG_FID']

print(f'Grid cells : {len(grid_file):,}')
print(f'US states  : {len(us_states)}')
print(f'Counties   : {len(counties_usa)}')
print('Spatial layers loaded')

Grid cells : 2,425
US states  : 51
Counties   : 51
Spatial layers loaded


## 4. Scenario Pairs

In [11]:
SCENARIO_PAIRS = [
    # ── Baseline temperature (TBL) ──
    ['1M_C400TBLH0',   '13M_C400TBLH1'],
    ['1M_C400TBLH0',   '25M_C400TBLH2'],
    ['1M_C400TBLH0',   '37M_C400TBLH3'],
    ['1M_C400TBLH0',   '49M_C400TBLH123'],
    # ── +1.5 C ──
    ['2M_C400T1.5H0',  '14M_C400T1.5H1'],
    ['2M_C400T1.5H0',  '26M_C400T1.5H2'],
    ['2M_C400T1.5H0',  '38M_C400T1.5H3'],
    ['2M_C400T1.5H0',  '50M_C400T1.5H123'],
    # ── +2.5 C ──
    ['3M_C400T2.5H0',  '15M_C400T2.5H1'],
    ['3M_C400T2.5H0',  '27M_C400T2.5H2'],
    ['3M_C400T2.5H0',  '39M_C400T2.5H3'],
    ['3M_C400T2.5H0',  '51M_C400T2.5H123'],
    # ── +4.0 C ──
    ['4M_C400T4H0',    '16M_C400T4H1'],
    ['4M_C400T4H0',    '28M_C400T4H2'],
    ['4M_C400T4H0',    '40M_C400T4H3'],
    ['4M_C400T4H0',    '52M_C400T4H123'],
]

print(f'{len(SCENARIO_PAIRS)} scenario pairs defined')

16 scenario pairs defined


## 5. Plotting Helpers

In [12]:
def draw_base_layers(ax):
    """Draw city mask, county borders, state borders, and study-area boundary."""
    city_mask.plot(ax=ax, facecolor=STYLE['city_face'],
                  edgecolor=STYLE['city_edge'],
                  linewidth=STYLE['lw_counties'], zorder=1, alpha=0.4)


def draw_overlays(ax):
    """Draw county/state borders and study-area boundary on top."""
    counties_usa.plot(ax=ax, facecolor='none',
                     edgecolor=STYLE['color_borders'],
                     linewidth=STYLE['lw_counties'], zorder=15)
    us_states.plot(ax=ax, facecolor='none',
                  edgecolor=STYLE['color_borders'],
                  linewidth=STYLE['lw_states'], zorder=15)
    temp_shp.plot(ax=ax, facecolor='none', edgecolor='black',
                 linewidth=STYLE['lw_boundary'], zorder=15)


def style_ax(ax, title):
    """Apply standard styling to a map axis."""
    ax.set_title(title, fontsize=14)
    ax.set_facecolor(STYLE['facecolor_bg'])
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    set_zoom(ax, ref_districts, STYLE['zoom_main'])


def plot_positive_dots(ax, pos_gdf):
    """Plot black centroid dots for cells where the stronger model is positive."""
    if pos_gdf.empty:
        return
    pts = pos_gdf.copy()
    pts['geometry'] = pts.centroid
    pts.plot(ax=ax, color='black', markersize=STYLE['markersize'],
             marker='o', linewidth=STYLE['marker_lw'], zorder=3)


def plot_colour_grid(ax, gdf, colour_col):
    """Plot the coloured grid cells."""
    gdf.plot(color=gdf[colour_col], edgecolor='black',
             linewidth=0.08, ax=ax, zorder=1)


print('Plotting helpers loaded')

Plotting helpers loaded


## 6. Main Loop — Mean Map Only

In [ ]:
for idx, (name_A, name_B) in enumerate(SCENARIO_PAIRS):

    label = f'{name_B} - {name_A}'
    print(f'\n{"="*60}')
    print(f'Pair {idx:>2d}/{len(SCENARIO_PAIRS)}:  {label}')
    print(f'{"="*60}')

    # ── 1. Process yield CSVs ──────────────────────────────
    csv_dssat_A  = process_yield_data(os.path.join(DSSAT_DIR,  name_A))
    csv_dssat_B  = process_yield_data(os.path.join(DSSAT_DIR,  name_B))
    csv_wofost_A = process_yield_data(os.path.join(WOFOST_DIR, name_A))
    csv_wofost_B = process_yield_data(os.path.join(WOFOST_DIR, name_B))

    if any(p is None for p in [csv_dssat_A, csv_dssat_B, csv_wofost_A, csv_wofost_B]):
        print('  Skipping — missing data')
        continue

    dssat_A  = grid_file.merge(pd.read_csv(csv_dssat_A),  left_on='id', right_on='real_id')
    dssat_B  = grid_file.merge(pd.read_csv(csv_dssat_B),  left_on='id', right_on='real_id')
    wofost_A = grid_file.merge(pd.read_csv(csv_wofost_A), left_on='id', right_on='real_id')
    wofost_B = grid_file.merge(pd.read_csv(csv_wofost_B), left_on='id', right_on='real_id')

    # ── 2. Compute diffs (heatwave - baseline) ────────────
    D = compute_diffs(dssat_B,  dssat_A)
    W = compute_diffs(wofost_B, wofost_A)

    D = D.rename(columns={'mean_percent_difference': 'mean_pct_D',
                          'mean_difference_STD':     'std_D'})
    W = W.rename(columns={'mean_percent_difference': 'mean_pct_W',
                          'mean_difference_STD':     'std_W'})

    # ── 3. Identify stronger model & positive-flag ────────
    merged = D[['real_id', 'geometry', 'mean_pct_D']].merge(
        W[['real_id', 'mean_pct_W']], on='real_id', how='inner'
    )
    merged['abs_D'] = merged['mean_pct_D'].abs()
    merged['abs_W'] = merged['mean_pct_W'].abs()
    merged['stronger'] = np.where(merged['abs_D'] > merged['abs_W'], 'D', 'W')
    merged['positive_flag'] = np.where(
        ((merged['stronger'] == 'D') & (merged['mean_pct_D'] > 0)) |
        ((merged['stronger'] == 'W') & (merged['mean_pct_W'] > 0)),
        'yes', 'no'
    )

    n_pos = (merged['positive_flag'] == 'yes').sum()
    print(f'  Grid cells: {len(merged):,}  |  Positive-flag: {n_pos}')

    # ── 4. Bivariate colour — MEAN ────────────────────────
    ref_mean = D['mean_pct_D'].replace(0, np.nan)
    dy_mean  = ((W['mean_pct_W'] - ref_mean) / ref_mean).clip(-1, 1)

    gdf_mean = D.copy()
    gdf_mean['color'] = rgb_to_hex_safe(dy_to_rgb(dy_mean.to_numpy()))

    merged = merged.to_crs(epsg=4326)
    gdf_pos = merged[merged['positive_flag'] == 'yes'].copy()

    # ── 5. Create single-panel figure (Mean only) ─────────
    fig, ax = plt.subplots(1, 1, figsize=STYLE['fig_size'])

    draw_base_layers(ax)
    plot_positive_dots(ax, gdf_pos)
    plot_colour_grid(ax, gdf_mean, 'color')
    draw_overlays(ax)
    style_ax(ax, f'{label}  mean')
    add_inset(ax, district_60, us_states, ref_districts, STYLE['zoom_inset'])

    # ── Save ──────────────────────────────────────────────
    out_path = os.path.join(OUTPUT_DIR, f'fig_11{label}.png')
    plt.savefig(out_path, dpi=STYLE['dpi'], bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {out_path}')

print(f'\nDone — {len(SCENARIO_PAIRS)} pairs processed.')


Pair  0/16:  13M_C400TBLH1 - 1M_C400TBLH0
  Grid cells: 2,423  |  Positive-flag: 2


C:\Users\yaron\AppData\Local\Temp\ipykernel_31192\3486665464.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pts['geometry'] = pts.centroid


  Saved: c:\BioCro_DSSAT_WOFOST_Egorov_etal_paper-main\data\output_maps_maize_fig11\fig_1113M_C400TBLH1 - 1M_C400TBLH0.png

Pair  1/16:  25M_C400TBLH2 - 1M_C400TBLH0
  Grid cells: 2,423  |  Positive-flag: 0
  Saved: c:\BioCro_DSSAT_WOFOST_Egorov_etal_paper-main\data\output_maps_maize_fig11\fig_1125M_C400TBLH2 - 1M_C400TBLH0.png

Pair  2/16:  37M_C400TBLH3 - 1M_C400TBLH0
  Grid cells: 2,423  |  Positive-flag: 0
  Saved: c:\BioCro_DSSAT_WOFOST_Egorov_etal_paper-main\data\output_maps_maize_fig11\fig_1137M_C400TBLH3 - 1M_C400TBLH0.png

Pair  3/16:  49M_C400TBLH123 - 1M_C400TBLH0
  Grid cells: 2,423  |  Positive-flag: 0
  Saved: c:\BioCro_DSSAT_WOFOST_Egorov_etal_paper-main\data\output_maps_maize_fig11\fig_1149M_C400TBLH123 - 1M_C400TBLH0.png

Pair  4/16:  14M_C400T1.5H1 - 2M_C400T1.5H0
  Grid cells: 2,423  |  Positive-flag: 0
  Saved: c:\BioCro_DSSAT_WOFOST_Egorov_etal_paper-main\data\output_maps_maize_fig11\fig_1114M_C400T1.5H1 - 2M_C400T1.5H0.png

Pair  5/16:  26M_C400T1.5H2 - 2M_C400T1.

C:\Users\yaron\AppData\Local\Temp\ipykernel_31192\3486665464.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pts['geometry'] = pts.centroid
